<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">


# Python for Finance, 3rd Edition
## Chapter 20 · Asset Management Systems and Reporting
&copy; Dr. Yves J. Hilpisch<br>
AI-supported by various LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


## Notebook Goals
This notebook mirrors the main Chapter 20 examples:
- build holdings and exposure snapshots,
- construct a compact performance and risk report versus `SPY`, and
- compute sector weights and linked contributions, including figures.


### Imports
We start with the core scientific Python stack and configure Matplotlib to
match the book style.


In [ ]:
from pathlib import Path
import subprocess
import sys

NOTEBOOK_SUBDIR = "notebooks"
COLAB_PACKAGES = {}
REPO_NAME = "py4fi3rd"
REPO_URL = "https://github.com/yhilpisch/py4fi3rd.git"


def _support_dir() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        support_dir = candidate / "notebooks"
        if (support_dir / "_book_notebook_support.py").exists():
            return support_dir
    if "google.colab" in sys.modules:
        root = Path("/content") / REPO_NAME
        if not root.exists():
            subprocess.run(
                ["git", "clone", "--depth", "1", REPO_URL, str(root)],
                check=True,
            )
        return root / "notebooks"
    raise RuntimeError("Could not locate notebook support helpers.")


SUPPORT_DIR = _support_dir()
if str(SUPPORT_DIR) not in sys.path:
    sys.path.insert(0, str(SUPPORT_DIR))

from _book_notebook_support import setup_notebook

CONTEXT = setup_notebook(
    notebook_subdir=NOTEBOOK_SUBDIR,
    colab_packages=COLAB_PACKAGES,
)

PROJECT_ROOT = CONTEXT["PROJECT_ROOT"]
NOTEBOOK_DIR = CONTEXT["NOTEBOOK_DIR"]
CODE_DIR = CONTEXT["CODE_DIR"]
CHAPTERS_DIR = CONTEXT["CHAPTERS_DIR"]
FIGURES_DIR = CONTEXT["FIGURES_DIR"]
DATA_DIR = CONTEXT["DATA_DIR"]

PROJECT_ROOT

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
# Matplotlib defaults for the book
mpl.style.use("seaborn-v0_8")
mpl.rcParams.update({"font.family": "serif"})
mpl.rcParams.update({"figure.dpi": 300})

### Holdings Snapshot and Basic Exposures
Recreate the small holdings table from the chapter and compute sector weights
as a first reporting view.


In [ ]:
holdings = pd.DataFrame(
    {
        "symbol": ["AAPL", "NVDA", "JPM", "TLT"],
        "quantity": [120, 80, 150, 1000],
        "price": [180.25, 820.10, 145.30, 102.50],
        "sector": [
            "Technology",
            "Technology",
            "Financials",
            "Fixed Income",
        ],
        "region": ["US", "US", "US", "US"],
        "currency": ["USD", "USD", "USD", "USD"],
    },
)
holdings["market_value"] = holdings["quantity"] * holdings["price"]
holdings["weight"] = (
    holdings["market_value"] / holdings["market_value"].sum()
)
holdings

In [ ]:
sector_weights = holdings.groupby("sector")["weight"].sum()
sector_weights

### Portfolio and Benchmark Returns
Load the end-of-day dataset, align it with the holdings universe, and
compute portfolio and benchmark returns over the last two years.


In [ ]:
LOCAL_EOD = DATA_DIR / "eod_data.csv"
if not LOCAL_EOD.exists():
    raise FileNotFoundError(LOCAL_EOD)

prices = pd.read_csv(
    LOCAL_EOD,
    parse_dates=["Date"],
    index_col="Date",
)

symbols = holdings["symbol"].tolist()
cols = symbols + ["SPY"]
sub = prices[cols].dropna(how="any")
sub = sub.iloc[-2 * 252 :]

quantities = holdings.set_index("symbol")["quantity"]
values = sub[symbols].mul(quantities, axis=1)
port_val = values.sum(axis=1)

r_port = port_val.pct_change().dropna()
r_bench = sub["SPY"].pct_change().dropna()
r_port.head(), r_bench.head()

### Performance and Risk Report
Compute annualized returns, volatility, max drawdown, tracking error, and the
information ratio for the portfolio versus `SPY`.


In [ ]:
ann_factor = 252.0
n_obs = len(r_port)

port_total = (1 + r_port).prod()
bench_total = (1 + r_bench).prod()
port_ann = port_total ** (ann_factor / n_obs) - 1
bench_ann = bench_total ** (ann_factor / n_obs) - 1

port_vol = r_port.std(ddof=1) * np.sqrt(ann_factor)
bench_vol = r_bench.std(ddof=1) * np.sqrt(ann_factor)

active = r_port - r_bench
te_ann = active.std(ddof=1) * np.sqrt(ann_factor)

def max_drawdown(returns):
    cum = pd.concat([pd.Series([1.0]), (1 + returns).cumprod()])
    running_max = cum.cummax()
    drawdowns = cum / running_max - 1
    return drawdowns.min()

report = pd.DataFrame(
    {
        "annualized_return": [port_ann, bench_ann],
        "annualized_volatility": [port_vol, bench_vol],
        "max_drawdown": [
            max_drawdown(r_port),
            max_drawdown(r_bench),
        ],
        "annualized_tracking_error": [te_ann, 0.0],
    },
    index=["Portfolio", "Benchmark (SPY)"],
)
report["information_ratio"] = [
    active.mean() * ann_factor / te_ann, np.nan
]
report.round(3)

### Compliance and Constraint Monitoring

Check position and sector caps and the tracking-error corridor.


In [ ]:
single_name_ok = holdings["weight"].le(0.50).all()

In [ ]:
sector_cap_ok = (
    sector_weights["Financials"] <= 0.25
)

In [ ]:
te_band_ok = 0.02 <= te_ann <= 0.06

In [ ]:
bool(single_name_ok), bool(sector_cap_ok), bool(te_band_ok)

### Ingesting and Validating Data

Load prices through a shared parsing and index-validation helper.


In [ ]:
from pathlib import Path
import pandas as pd

In [ ]:
def load_prices(path):
    prices = pd.read_csv(
        path,
        parse_dates=["Date"],
        index_col="Date",
    )
    prices = prices.dropna(how="any")
    assert prices.index.is_monotonic_increasing
    return prices

In [ ]:
prices = load_prices("../data/eod_data.csv")

In [ ]:
prices.index.min(), prices.index.max()

### Running Analytics and Storing Results

Collect daily returns and persist the analytics table for reuse.


In [ ]:
analytics = pd.DataFrame(
    {
        "r_port": r_port,
        "r_bench": r_bench,
        "active": active,
    }
)

In [ ]:
analytics.head()

In [ ]:
Path("../data/analytics").mkdir(parents=True, exist_ok=True)
analytics.to_parquet(
    "../data/analytics/ch20_portfolio_vs_spy.parquet"
)

### Scheduling and Failure Handling

Dispatch named reporting jobs with explicit success and error messages.


In [ ]:
JOBS = {
    "daily_analytics": lambda: analytics,  # placeholder
    "monthly_client_report": lambda: report,
}

In [ ]:
def run_job(name):
    try:
        result = JOBS[name]()
    except Exception as exc:
        print(f"[ERROR] Job {name} failed: {exc}")
        raise
    else:
        print(f"[OK] Job {name} finished")
        return result

In [ ]:
run_job("daily_analytics")

### Sector Weights and Linked Contributions
Estimate average sector weights and linked sector contributions to realized
performance over the same window.


In [ ]:
rets_assets = sub.pct_change().dropna()
start_values = values.shift(1).loc[rets_assets.index]
weights = start_values.div(start_values.sum(axis=1), axis=0)

sector_map = holdings.set_index("symbol")["sector"]
sector_weights_time = weights.T.groupby(sector_map).sum().T
sector_weights_avg = sector_weights_time.mean()
sector_weights_avg

In [ ]:
daily_contrib = weights * rets_assets
daily_port = daily_contrib.sum(axis=1)
growth = (1.0 + daily_port).cumprod()
future_growth = float(growth.iloc[-1]) / growth
linked = daily_contrib.mul(future_growth, axis=0).sum()
sector_contrib = linked.groupby(sector_map).sum()
sector_contrib

### Figures
Plot cumulative performance for the portfolio and benchmark and visualize
sector weights and contributions.


In [ ]:
aligned = pd.concat(
    {"portfolio": r_port, "benchmark": r_bench},
    axis=1,
).dropna()
cum_port = (1 + aligned["portfolio"]).cumprod()
cum_bench = (1 + aligned["benchmark"]).cumprod()

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(cum_port.index, cum_port.values, label="Portfolio")
ax.plot(cum_bench.index, cum_bench.values,
        label="Benchmark (SPY)")
ax.set_ylabel("Cumulative value (start = 1.0)")
ax.set_title("Cumulative portfolio vs benchmark performance")
locator = mdates.AutoDateLocator()
formatter = mdates.ConciseDateFormatter(locator)
ax.xaxis.set_major_locator(locator)
ax.xaxis.set_major_formatter(formatter)
ax.grid(True, linestyle="--", alpha=0.3)
ax.legend(loc="best")
fig.autofmt_xdate()
fig.tight_layout()

In [ ]:
sectors = sector_weights_avg.index.tolist()
x = np.arange(len(sectors))

fig, axes = plt.subplots(1, 2, figsize=(8, 4), sharex=True)
axes[0].bar(x, sector_weights_avg.values, color="tab:blue")
axes[0].set_ylabel("Average sector weight")
axes[0].set_title("Average sector weights")
axes[0].grid(True, axis="y", linestyle="--", alpha=0.3)

axes[1].bar(x, sector_contrib.values, color="tab:green")
axes[1].set_ylabel("Cumulative return contribution")
axes[1].set_title("Sector contributions to performance")
axes[1].grid(True, axis="y", linestyle="--", alpha=0.3)

for ax in axes:
    ax.set_xticks(x)
    ax.set_xticklabels(sectors, rotation=20, ha="right")

fig.tight_layout()

### Configuration, Code, and Data Lineage

Record the inputs and code behind the report.


In [ ]:
report_metadata = {
    "as_of": analytics.index.max().strftime("%Y-%m-%d"),
    "config_file": "inline-example",
    "code_module": (
        "code.chapters."
        + "ch20_asset_management_systems_and_reporting"
    ),
    "data_source": "data/eod_data.csv",
}

In [ ]:
report_metadata

### Controls and Approvals

Represent review steps, owners, and completion status as data.


In [ ]:
import pandas as pd

In [ ]:
checks = pd.DataFrame(
    {
        "step": [
            "Verify holdings vs custodian",
            "Validate performance vs independent source",
            "Review attribution commentary",
        ],
        "owner": ["Ops", "Risk", "PM"],
        "status": ["done", "done", "pending"],
    }
)

In [ ]:
checks

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">
